In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, accuracy_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder

# 1. Chargement des données (Garde tout le "bruit" et les doublons)
train_data = pd.read_csv('conversion_data_train.csv')
test_data = pd.read_csv('conversion_data_test.csv')

# 2. Préparation des variables
target = 'converted'
X = train_data.drop(target, axis=1)
y = train_data[target]

# Encodage des catégories (Country, Source)
categorical_cols = ['country', 'source']
X_encoded = X.copy()
X_test_encoded = test_data.copy()
cat_indices = [X.columns.get_loc(col) for col in categorical_cols]

for col in categorical_cols:
    le = LabelEncoder()
    # On fit sur l'ensemble pour éviter les catégories inconnues
    le.fit(pd.concat([X[col], test_data[col]]))
    X_encoded[col] = le.transform(X[col])
    X_test_encoded[col] = le.transform(test_data[col])

# 3. Split Stratifié (pour respecter les 3% de conversion)
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Entraînement du Monstre Poisson
# HistGradientBoostingRegressor avec loss='poisson' est idéal pour les événements rares
model = HistGradientBoostingRegressor(
    loss='poisson',
    learning_rate=0.05,
    max_iter=500,
    max_depth=8,
    l2_regularization=0.1,
    categorical_features=cat_indices,
    random_state=42
)

print("🚀 Entraînement du modèle Poisson...")
model.fit(X_train, y_train)

# 5. Prédiction des scores et Optimisation du Seuil pour le F1
y_scores = model.predict(X_val)

best_f1 = 0
best_threshold = 0
thresholds = np.linspace(y_scores.min(), y_scores.max(), 300)

for t in thresholds:
    preds = (y_scores >= t).astype(int)
    score = f1_score(y_val, preds)
    if score > best_f1:
        best_f1 = score
        best_threshold = t

# 6. Affichage des métriques finales
y_preds = (y_scores >= best_threshold).astype(int)
print(f"\n--- RÉSULTATS POISSON MONSTER ---")
print(f"Seuil Optimal : {best_threshold:.6f}")
print(f"F1-Score      : {best_f1:.5f}")
print(f"ROC-AUC       : {roc_auc_score(y_val, y_scores):.5f}")
print(f"Précision     : {precision_score(y_val, y_preds):.5f}")
print(f"Recall        : {recall_score(y_val, y_preds):.5f}")

# 7. Génération du fichier de soumission
test_scores = model.predict(X_test_encoded)
test_preds = (test_scores >= best_threshold).astype(int)

submission = pd.DataFrame({'converted': test_preds})
submission.to_csv('submission_POISSON_MONSTER.csv', index=False)
print(f"\n✅ Fichier 'submission_POISSON_MONSTER.csv' généré ({test_preds.sum()} conversions).")

🚀 Entraînement du modèle Poisson...

--- RÉSULTATS POISSON MONSTER ---
Seuil Optimal : 0.404879
F1-Score      : 0.77128
ROC-AUC       : 0.98598
Précision     : 0.82324
Recall        : 0.72549

✅ Fichier 'submission_POISSON_MONSTER.csv' généré (893 conversions).
